# Processing SST1 RSoXS Data

## Pip install and restart kernel 

In [ ]:
# Only needs to be run once per jupyterhub session, **restart kernel after running**

# !pip install pyhyperscattering  # to use pip published package
!pip install -e /nsls2/users/alevin/repos/pyhyper_toneygroup_fork/PyHyperScattering  # to use pip to install via directory

## Imports

In [ ]:
## The autoreload IPython magic command reloads all modules before code is ran
%load_ext autoreload
%autoreload

In [ ]:
## Imports
import PyHyperScattering as phs
import pathlib
import pickle
import sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib_inline.backend_inline import set_matplotlib_formats

sys.path.append('/nsls2/users/alevin/local_lib')
from andrew_rsoxs_fxns import *

## Some setup functions
set_matplotlib_formats('svg')
# c = from_profile('rsoxs')
print(f'Using PyHyperScattering Version: {phs.__version__}')
rsoxsload = phs.load.SST1RSoXSDB(corr_mode='None', use_chunked_loading=True)  # initialize rsoxs databroker loader w/ Dask

## Define masks directory path
basePath = pathlib.Path('/nsls2/users/alevin')
maskPath = basePath.joinpath('masks')

## Set an RSoXS colormap for later
cm = plt.cm.terrain.copy()
cm.set_bad('purple')

## Loading raw data from databroker

In [ ]:
# ## Search for and summarize runs:
# runs_sum_df = rsoxsload.summarize_run(institution='CUBLDER', plan='full_carbon_scan_nd')
# display(runs_sum_df)

## Slice output dataframe for samples of interest
runs_of_interest = runs_sum_df.loc[runs_sum_df['cycle']=='2022-2'].loc[runs_sum_df['sample_id']=='andrew1']
scans = list(runs_of_interest['scan_id'])
display(runs_of_interest)

In [ ]:
raw_saxs = load_stacked_pol(rsoxsload, scans[0], scans[1])
# raw_saxs = raw_saxs.drop_vars('dark_id')

raw_waxs = load_stacked_pol(rsoxsload, scans[2], scans[3])
# raw_waxs = raw_waxs.drop_vars('dark_id')
display(raw_saxs, raw_waxs)

## Draw/check masks & beamcenters for transforming to q-space
#### 1. Check raw images for all images:

In [ ]:
saxs_waxs_p00_p90_plot(raw_saxs, raw_waxs)

#### 2. Draw 

#### - If you need to draw new masks:

In [ ]:
# ## SAXS:
# saxs_mask_img = raw_saxs.sel(pol=0, energy=280, method='nearest').compute()
# draw = phs.IntegrationUtils.DrawMask(saxs_mask_img)
# draw.ui()

In [ ]:
## Save and load saxs drawn mask
draw.save(maskPath.joinpath(f'saxs_{raw_saxs.sample_name}.json'))
saxs_mask = draw.mask

In [ ]:
# ## Repeat for WAXS mask:
# waxs_mask_img = raw_waxs.sel(pol=0, energy=280, method='nearest').compute()
# draw = phs.IntegrationUtils.DrawMask(waxs_mask_img)
# draw.ui()

In [ ]:
# ## Save and load saxs drawn mask
# draw.save(maskPath.joinpath(f'waxs_{raw_waxs.sample_name}.json'))
# waxs_mask = draw.mask

#### - Or load previously drawn masks:

In [ ]:
# ## SAXS:
saxs_mask_img = raw_saxs.sel(pol=0, energy=275, method='nearest').compute()
# draw = phs.IntegrationUtils.DrawMask(saxs_mask_img)
draw.load(maskPath.joinpath(f'saxs_{raw_saxs.sample_name}.json'))
saxs_mask = draw.mask

# ## WAXS: 
waxs_mask_img = raw_waxs.sel(pol=0, energy=275, method='nearest').compute()
# draw = phs.IntegrationUtils.DrawMask(waxs_mask_img)
# draw.load(maskPath.joinpath(f'waxs_{raw_waxs.sample_name}.json'))
# waxs_mask = draw.mask




In [ ]:
# ## Check both masks:
# fig, axs = plt.subplots(nrows=1, ncols=2)
# fig.set(tight_layout=True, size_inches=(8,4))
# axs[0].imshow(saxs_mask, origin='lower')
# axs[0].set(title='SAXS mask', xlabel='pix_x', ylabel='pix_y')
# axs[1].imshow(waxs_mask, origin='lower')
# axs[1].set(title='WAXS mask', xlabel='pix_x', ylabel='pix_y')
# plt.show()
# plt.close('all')

#### 3. Check beamcenter before converting to q-space

In [ ]:
## SAXS
SAXSinteg = phs.integrate.PFEnergySeriesIntegrator(geomethod='template_xr', template_xr = raw_saxs.sel(pol=0))
# SAXSinteg.mask = saxs_mask
SAXSinteg.ni_beamcenter_x = correct_bcxy_2022_2['saxs_bcx']
SAXSinteg.ni_beamcenter_y = correct_bcxy_2022_2['saxs_bcy']
raw_saxs.attrs['beamcenter_x'] = correct_bcxy_2022_2['saxs_bcx']
raw_saxs.attrs['beamcenter_y'] = correct_bcxy_2022_2['saxs_bcy']
raw_saxs.attrs['poni1'] = SAXSinteg.poni1
raw_saxs.attrs['poni2'] = SAXSinteg.poni2
print('SAXS Beamcenter: \n'
      f'poni1: {SAXSinteg.poni1}, poni2: {SAXSinteg.poni2} \n'
      f'ni_beamcenter_y: {SAXSinteg.ni_beamcenter_y}, ni_beamcenter_x: {SAXSinteg.ni_beamcenter_x}')

## Plot check
phs.IntegrationUtils.Check.checkAll(SAXSinteg, saxs_mask_img, img_max=1e3, alpha=0.6)
plt.xlim(SAXSinteg.ni_beamcenter_x-200, SAXSinteg.ni_beamcenter_x+200)
plt.ylim(SAXSinteg.ni_beamcenter_y-200, SAXSinteg.ni_beamcenter_y+200)
plt.gcf().set(dpi=120)
plt.show()

## WAXS
WAXSinteg = phs.integrate.PFEnergySeriesIntegrator(geomethod='template_xr', template_xr = raw_waxs.sel(pol=0))
# WAXSinteg.mask = waxs_mask
WAXSinteg.ni_beamcenter_x = correct_bcxy_2022_2['waxs_bcx']
WAXSinteg.ni_beamcenter_y = correct_bcxy_2022_2['waxs_bcy']
raw_waxs.attrs['beamcenter_x'] = correct_bcxy_2022_2['waxs_bcx']
raw_waxs.attrs['beamcenter_y'] = correct_bcxy_2022_2['waxs_bcy']
raw_waxs.attrs['poni1'] = WAXSinteg.poni1
raw_waxs.attrs['poni2'] = WAXSinteg.poni2
print('WAXS Beamcenter: \n'
      f'poni1: {WAXSinteg.poni1}, poni2: {WAXSinteg.poni2} \n'
      f'ni_beamcenter_y: {WAXSinteg.ni_beamcenter_y}, ni_beamcenter_x: {WAXSinteg.ni_beamcenter_x}')

## Plot check
phs.IntegrationUtils.Check.checkAll(WAXSinteg, waxs_mask_img, img_max=7e3, alpha=0.6)
plt.xlim(WAXSinteg.ni_beamcenter_x-200, WAXSinteg.ni_beamcenter_x+200)
plt.ylim(WAXSinteg.ni_beamcenter_y-200, WAXSinteg.ni_beamcenter_y+200)
plt.gcf().set(dpi=120)
plt.show()

In [ ]:
# ## Using Pete D.'s (very slightly modified) beamcentering script:
# # phs.BeamCentering.CenteringAccessor.refine_geometry

# ## SAXS
# res_saxs = raw_saxs.sel(pol=0).util.refine_geometry(energy=275, q_min=0.002, q_max=0.006)
# # res_saxs = raw_saxs.sel(pol=0).util.refine_geometry(energy=275, q_min=0.002, q_max=0.006, chi_min=-180, chi_max=60)
# # res_saxs = raw_saxs.sel(pol=0).util.refine_geometry(energy=280, q_min=0.002, q_max=0.008, mask=saxs_mask)
# raw_saxs.attrs['poni1'] = res_saxs.x[0]
# raw_saxs.attrs['poni2'] = res_saxs.x[1]
# SAXSinteg = phs.integrate.PFEnergySeriesIntegrator(geomethod='template_xr', template_xr = raw_saxs.sel(pol=0))
# SAXSinteg.mask = saxs_mask

# ## SAXS Plot check
# print('SAXS Beamcenter Post-optimization: \n'
#       f'poni1: {SAXSinteg.poni1}, poni2: {SAXSinteg.poni2} \n'
#       f'ni_beamcenter_y: {SAXSinteg.ni_beamcenter_y}, ni_beamcenter_x: {SAXSinteg.ni_beamcenter_x}')
# phs.IntegrationUtils.Check.checkAll(SAXSinteg, saxs_mask_img, img_max=1e3, alpha=0.6)
# plt.xlim(SAXSinteg.ni_beamcenter_x-200, SAXSinteg.ni_beamcenter_x+200)
# plt.ylim(SAXSinteg.ni_beamcenter_y-200, SAXSinteg.ni_beamcenter_y+200)
# plt.gcf().set(dpi=120)
# plt.show()

# ## WAXS
# # res_waxs = raw_waxs.sel(pol=0).util.refine_geometry(energy=275, q_min=0.02, q_max=0.06, chi_min=-10, chi_max=70)
# res_waxs = raw_waxs.sel(pol=0).util.refine_geometry(energy=275, q_min=0.02, q_max=0.06)
# raw_waxs.attrs['poni1'] = res_waxs.x[0]
# raw_waxs.attrs['poni2'] = res_waxs.x[1]
# WAXSinteg = phs.integrate.PFEnergySeriesIntegrator(geomethod='template_xr', template_xr = raw_waxs.sel(pol=0))
# WAXSinteg.mask = waxs_mask

# ## WAXS Plot check
# print('WAXS Beamcenter Post-optimization: \n'
#       f'poni1: {WAXSinteg.poni1}, poni2: {WAXSinteg.poni2} \n'
#       f'ni_beamcenter_y: {WAXSinteg.ni_beamcenter_y}, ni_beamcenter_x: {WAXSinteg.ni_beamcenter_x}')
# phs.IntegrationUtils.Check.checkAll(WAXSinteg, waxs_mask_img, img_max=5e3, alpha=0.6)
# plt.xlim(WAXSinteg.ni_beamcenter_x-200, WAXSinteg.ni_beamcenter_x+200)
# plt.ylim(WAXSinteg.ni_beamcenter_y-200, WAXSinteg.ni_beamcenter_y+200)
# plt.gcf().set(dpi=120)
# plt.show()

# ## Tweaking if needed:
# ## SAXS Tweaking & Plot Check
# saxs_new_bcx = 488
# saxs_new_bcy = 515
# SAXSinteg.ni_beamcenter_x = saxs_new_bcx
# raw_saxs.attrs['beamcenter_x'] = saxs_new_bcx
# SAXSinteg.ni_beamcenter_y = saxs_new_bcy
# raw_saxs.attrs['beamcenter_y'] = saxs_new_bcy

# print('SAXS Beamcenter Tweaking: \n'
#       f'poni1: {SAXSinteg.poni1}, poni2: {SAXSinteg.poni2} \n'
#       f'ni_beamcenter_y: {SAXSinteg.ni_beamcenter_y}, ni_beamcenter_x: {SAXSinteg.ni_beamcenter_x}')

# phs.IntegrationUtils.Check.checkAll(SAXSinteg, saxs_mask_img, img_max=1e3, alpha=0.6)
# plt.xlim(SAXSinteg.ni_beamcenter_x-200, SAXSinteg.ni_beamcenter_x+200)
# plt.ylim(SAXSinteg.ni_beamcenter_y-200, SAXSinteg.ni_beamcenter_y+200)
# plt.gcf().set(dpi=120)
# plt.show()

# ## WAXS Tweaking & Plot Check
# waxs_new_bcx = 396.3
# waxs_new_bcy = 553
# WAXSinteg.ni_beamcenter_x = waxs_new_bcx
# raw_waxs.attrs['beamcenter_x'] = waxs_new_bcx
# WAXSinteg.ni_beamcenter_y = waxs_new_bcy
# raw_waxs.attrs['beamcenter_x'] = waxs_new_bcx

# print('WAXS Beamcenter Tweaking: \n'
#       f'poni1: {WAXSinteg.poni1}, poni2: {WAXSinteg.poni2} \n'
#       f'ni_beamcenter_y: {WAXSinteg.ni_beamcenter_y}, ni_beamcenter_x: {WAXSinteg.ni_beamcenter_x}')
# phs.IntegrationUtils.Check.checkAll(WAXSinteg, waxs_mask_img, img_max=5e3, alpha=0.6, guide1=40)
# plt.xlim(WAXSinteg.ni_beamcenter_x-200, WAXSinteg.ni_beamcenter_x+200)
# plt.ylim(WAXSinteg.ni_beamcenter_y-200, WAXSinteg.ni_beamcenter_y+200)
# plt.gcf().set(dpi=120)
# plt.show()

## Convert to q-space:

In [ ]:
### Now that we know our beamcenters are accurate, we can apply correct q axis labels
raw_waxs = apply_q_labels(raw_waxs)
raw_saxs = apply_q_labels(raw_saxs)

# ## 4 Images (SAXS & WAXS * 0 & 90 pol):
# fig, ((sax00, sax90), (wax00, wax90)) = plt.subplots(nrows=2, ncols=2, subplot_kw=(dict()))
# fig.set(tight_layout=True, size_inches=(8,8))

# energy=280

# raw_saxs.sel(pol=0, energy=energy, method='nearest').plot.imshow(x='qx', y='qy', ax=sax00, origin='lower', norm=LogNorm(1e1, 5e3), cmap=cm, interpolation='antialiased', add_colorbar=False)
# sax00.set(aspect='equal')
# raw_saxs.sel(pol=90, energy=energy, method='nearest').plot.imshow(x='qx', y='qy', ax=sax90, origin='lower', norm=LogNorm(1e1, 5e3), cmap=cm, interpolation='antialiased', add_colorbar=False)
# sax90.set(aspect='equal')

# raw_waxs.sel(pol=0, energy=energy, method='nearest').plot.imshow(x='qx', y='qy', ax=wax00, origin='lower', norm=LogNorm(1e1, 5e3), cmap=cm, interpolation='antialiased', add_colorbar=False)
# wax00.set(aspect='equal')
# raw_waxs.sel(pol=90, energy=energy, method='nearest').plot.imshow(x='qx', y='qy', ax=wax90, origin='lower', norm=LogNorm(1e1, 5e3), cmap=cm, interpolation='antialiased', add_colorbar=False)
# wax90.set(aspect='equal')

# plt.show()
# plt.close('all')

#### 1. Cake image stacks

In [ ]:
integ_waxs = integrate_stacked_pol(WAXSinteg, raw_waxs)
integ_saxs = integrate_stacked_pol(SAXSinteg, raw_saxs)
display(integ_saxs, integ_waxs)

In [ ]:
# ### Compute full datasets?

# integ_waxs.data = integ_waxs.data.compute()
# integ_saxs.data = integ_saxs.data.compute()

## Analysis / related

In [ ]:
plt.rcParams['savefig.facecolor'] = 'white'
savePath = notebookPath.joinpath('exports_v2')

sample_name = sample_guide[raw_waxs.sample_name]
scan_id = raw_waxs.sampleid
detector = detector_guide[raw_waxs.detector]

scanPath = savePath.joinpath(f'{scan_id}_{sample_name}_{detector}_{int(pol):0>2}deg')
scanPath.mkdir(parents=True, exist_ok=True)

In [ ]:
energies = raw_waxs.energy.data
energy_list = energies[16:96]

# lineplot_cmap = plt.cm.viridis(np.linspace(0,0.9, len(energy_list)))

energy_list.shape

### 2D chi vs q map at select energy

In [ ]:
integ_waxs.sel(pol=0, energy=285, method='nearest').plot.imshow(
    norm=LogNorm(1e1, 5e3), cmap=cm, interpolation='antialiased', xlim=(0.01, 0.1))

### Facet plots

In [ ]:
### Facet plots:

# energy_list =

# integ_saxs.sel(pol=0, energy=energy_list).sel(q=slice(0.0008, 0.0101)).plot.imshow(
#     norm=LogNorm(1e1, 1e3), cmap=cm, interpolation='antialiased', col='energy', col_wrap=4)

# integ_waxs.sel(pol=0, energy=energy_list).sel(q=slice(0.01, 0.1)).plot.imshow(
#     norm=LogNorm(1e1, 5e3), cmap=cm, interpolation='antialiased', col='energy', col_wrap=4)

# raw_waxs.sel(pol=0, energy=np.linspace(280, 298, 20), method='nearest').plot.imshow(
#     norm=LogNorm(1e1, 5e3), cmap=cm, interpolation='antialiased', col='energy', col_wrap=4)



for num in range(10):
    pol = 90
    grid = raw_waxs.sel(pol=pol, energy=energy_list[8*num:8*num+8], method='nearest').plot.imshow(x='qx', y='qy',
                norm=LogNorm(1e1, 5e3), cmap=cm, interpolation='antialiased', col='energy', col_wrap=4)
    grid.set_xlabels('qx [1/Å]')
    grid.set_ylabels('qy [1/Å]') 
   
    sample_name = sample_guide[raw_waxs.sample_name]
    scan_id = raw_waxs.sampleid
    detector = detector_guide[raw_waxs.detector]

    # Create/select folder for scan to save plots:
    tiffsPath = scanPath.joinpath(f'facet_qxqy_frames_{detector}_{int(pol):0>2}deg')
    tiffsPath.mkdir(parents=True, exist_ok=True)
    
    plt.savefig(tiffsPath.joinpath(f'{sample_name}_{detector}_{int(pol):0>2}_f{num}.svg'))

### 1D Intensity Plots

In [ ]:
# 360 chi mean
pol=0
for i, energy in enumerate(energy_list):
    integ_waxs.sel(pol=pol, energy=energy, method='nearest').mean('chi').plot.line(label=round(energy,1), color=lineplot_cmap[i])

plt.title(f'I vs Q, 360 chi avg: {sample_guide[integ_waxs.sample_name]}, {detector_guide[integ_waxs.detector]}, pol={pol}')
plt.xscale('log')
plt.xlabel('Q [1/nm]')
plt.xlim((0.01, 0.1))
plt.yscale('log')
plt.ylabel('Intensity [arb. units]')
plt.ylim((1, 1e4))
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

In [ ]:
# Para chi mean
pol=0
for i, energy in enumerate(energy_list):
    integ_waxs.sel(pol=pol, energy=energy, method='nearest').sel(chi=slice(-10, 10)).mean('chi').plot.line(label=round(energy,1), color=lineplot_cmap[i])

plt.title(f'I vs Q, para chi avg: {sample_guide[integ_waxs.sample_name]}, {detector_guide[integ_waxs.detector]}, pol={pol}')
plt.xscale('log')
plt.xlabel('Q [1/nm]')
plt.xlim((0.01, 0.1))
plt.yscale('log')
plt.ylabel('Intensity [arb. units]')
plt.ylim((1, 1e4))
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

In [ ]:
# Perp chi mean
pol=0
for i, energy in enumerate(energy_list):
    integ_waxs.sel(pol=pol, energy=energy, method='nearest').sel(chi=slice(80, 100)).mean('chi').plot.line(label=round(energy,1), color=lineplot_cmap[i])

plt.title(f'I vs Q, perp chi avg: {sample_guide[integ_waxs.sample_name]}, {detector_guide[integ_waxs.detector]}, pol={pol}')
plt.xscale('log')
plt.xlabel('Q [1/nm]')
plt.xlim((0.01, 0.1))
plt.yscale('log')
plt.ylabel('Intensity [arb. units]')
plt.ylim((1, 1e4))
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

In [ ]:
# Perp chi mean
pol=0
for i, energy in enumerate(energy_list):
    integ_saxs.sel(pol=pol, energy=energy, method='nearest').sel(chi=slice(50, 130)).mean('chi').plot.line(label=round(energy,1), color=lineplot_cmap[i])

plt.title(f'I vs Q, perp chi avg: {sample_guide[integ_saxs.sample_name]}, {detector_guide[integ_saxs.detector]}, pol={pol}')
plt.xscale('log')
plt.xlabel('Q [1/nm]')
plt.xlim((0.002, 0.01))
plt.yscale('log')
plt.ylabel('Intensity [arb. units]')
plt.ylim((1, 2e2))
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

In [ ]:
# integ_waxs.sel(pol=0, energy=energy_list).mean('chi').plot.line(hue='energy', xscale='log', yscale='log', 
#     xlim=(0.01, 0.1), add_legend=False)
# plt.legend(labels=np.round(energy_list, 1))

### Anisotropy Ratios

In [ ]:
integ_saxs.rsoxs.AR()

In [ ]:
pol=0
AR_saxs_p0 = integ_saxs.sel(pol=pol).rsoxs.AR(pol=pol, chi_width=40)
AR_saxs_p0.plot.imshow(x='q', xscale='log', xlim=(0.002, 0.009), xticks=[0.002, 0.01], y='energy', 
                       ylim=(280, 290), cmap=plt.cm.seismic, vmin=-1, vmax=1, interpolation='antialiased', size=6, 
                       aspect=0.6)

plt.xlabel("q [1/Å]")
plt.title(f'AR map, E vs Q: {sample_guide[integ_saxs.sample_name]}, {detector_guide[integ_saxs.detector]}, pol={pol}')

In [ ]:
### Using pcolormesh instead of imshow:

pol=0
AR_waxs_p0 = integ_waxs.sel(pol=pol).rsoxs.AR(pol=pol, chi_width=40)
AR_waxs_p0.plot.pcolormesh(cmap=plt.cm.seismic, size=6, aspect=1, x='q', xscale='log', xlim=(0.01, 0.1),
                          vmin=-1, vmax=1, y='energy', ylim=(282, 290), infer_intervals=True)

plt.xlabel("q [1/Å]")
plt.title(f'AR map, E vs Q: {sample_guide[integ_waxs.sample_name]}, {detector_guide[integ_waxs.detector]}, pol={pol}')
plt.savefig(scanPath.joinpath(f'AR_map_{sample_name}_{detector}_pol{int(pol)}.svg'))

In [ ]:
pol=0
AR_waxs_p0 = integ_waxs.sel(pol=pol).rsoxs.AR(pol=pol, chi_width=40)
AR_waxs_p0.plot.imshow(x='q', xscale='log', xlim=(0.01, 0.09), xticks=[0.01, 0.1], y='energy', ylim=(280, 290),
                       cmap=plt.cm.seismic, vmin=-1, vmax=1, size=6, aspect=0.6)

plt.xlabel("q [1/Å]")
plt.title(f'AR map, E vs Q: {sample_guide[integ_waxs.sample_name]}, {detector_guide[integ_waxs.detector]}, pol={pol}')

In [ ]:
pol=90
AR_saxs_p90 = integ_saxs.sel(pol=pol).rsoxs.AR(pol=pol, chi_width=40)
AR_saxs_p90.plot.imshow(x='q', xscale='log', xlim=(0.002, 0.009), xticks=[0.01, 0.1], y='energy', ylim=(280, 290),
                       cmap=plt.cm.seismic, vmin=-1, vmax=1, interpolation='antialiased', size=6, aspect=0.6)

plt.xlabel("q [1/Å]")
plt.title(f'AR map, E vs Q: {sample_guide[integ_saxs.sample_name]}, {detector_guide[integ_saxs.detector]}, pol={pol}')

In [ ]:
pol=90
AR_waxs_p90 = integ_waxs.sel(pol=pol).rsoxs.AR(pol=pol, chi_width=40)
AR_waxs_p90.plot.imshow(x='q', xscale='log', xlim=(0.01, 0.09), xticks=[0.01, 0.1], y='energy', ylim=(280, 290),
                       cmap=plt.cm.seismic, vmin=-1, vmax=1, interpolation='antialiased', size=6, aspect=0.6)

plt.xlabel("q [1/Å]")
plt.title(f'AR map, E vs Q: {sample_guide[integ_waxs.sample_name]}, {detector_guide[integ_waxs.detector]}, pol={pol}')

In [ ]:
para_waxs_p0 = integ_waxs.sel(chi=slice(-10, 10)).mean('chi').sel(pol=0)
perp_waxs_p0 = integ_waxs.sel(chi=slice(80, 100)).mean('chi').sel(pol=0)
AR_waxs_p90 = (para_waxs_p0-perp_waxs_p0) / (para_waxs_p0+para_waxs_p0)
AR_waxs_p90

In [ ]:
plt.imshow(AR_waxs_p90.data, cmap=plt.cm.seismic, vmin=-1, vmax=1, interpolation='antialiased', origin='lower')
# plt.xscale('log')
# plt.gcf().set(dpi=300)

In [ ]:
AR_waxs_p90.plot(x='q', vmin=-0.8, vmax=0.8, cmap=plt.cm.seismic)
# AR_waxs_p90.plot.imshow(x='q', xlim=(0.01, 0.1), xscale='log',cmap=plt.cm.seismic, vmin=-1, 
#                 vmax=1, interpolation='none')
plt.ylim(275, 300)

In [ ]:
out90 = integ_waxs.sel(pol=90).rsoxs.AR(pol=90)
out90.plot.imshow(x='q', xlim=(0.01, 0.1), xscale='log', y='energy', ylim=(275, 290),cmap=plt.cm.seismic, vmin=-1, 
                vmax=1, interpolation='antialiased')